# Chapter 7 — Optimization for Deep Learning

Runnable companion to `docs/06_optimization.md`. Four short experiments on
a small MLP / MNIST-subset task — fast enough to run all of them under a
minute on CPU.

1. **Optimizer head-to-head** — SGD, SGD + momentum, Adam, AdamW.
2. **LR-range test** — find the largest stable learning rate by sweeping.
3. **Scheduler ablation** — StepLR vs. CosineAnnealing vs. OneCycle.
4. **Gradient clipping** — what `clip_grad_norm_` actually does to a
   too-large gradient.

Generated by `src/build_chapter_06_optimization_adam_sgd_scheduler.py`.

## Setup

In [1]:
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cpu")

from pathlib import Path as _Path
_ROOT = _Path.cwd()
while not (_ROOT / "ROADMAP.md").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
MNIST_ROOT = str(_ROOT / "datasets" / "mnist")
print("torch", torch.__version__, "device", DEVICE)
print("mnist root:", MNIST_ROOT)

torch 2.12.0+cu130 device cpu
mnist root: /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/datasets/mnist


## Small MNIST subset (shared across all experiments)

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
mnist_train = datasets.MNIST(MNIST_ROOT, train=True,  download=True, transform=transform)
mnist_val   = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(mnist_train, list(range(4000))),
                          batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(Subset(mnist_val,   list(range(2000))),
                          batch_size=256, shuffle=False, num_workers=0)
print(f"train: 4000, val: 2000")

train: 4000, val: 2000


## Helpers

In [3]:
def make_mlp():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 128), nn.ReLU(inplace=True),
        nn.Linear(128, 10),
    )


def train_epochs(model, optimizer, scheduler=None, num_epochs=5,
                 step_scheduler="epoch", clip_grad=None):
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_acc": [], "lr": []}
    for _ in range(num_epochs):
        model.train()
        running = 0.0; n = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            if clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            if scheduler is not None and step_scheduler == "batch":
                scheduler.step()
            running += loss.item() * x.size(0); n += x.size(0)
        if scheduler is not None and step_scheduler == "epoch":
            scheduler.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item(); total += y.size(0)
        history["train_loss"].append(running / n)
        history["val_acc"].append(correct / total)
        history["lr"].append(optimizer.param_groups[0]["lr"])
    return history

## 1. Optimizer head-to-head

Same MLP, same LR, four optimizers. Adam-family typically converges faster;
SGD + momentum often catches up but takes more epochs.

In [4]:
def fresh_model():
    torch.manual_seed(0)
    return make_mlp().to(DEVICE)


configs = {
    "SGD":         lambda m: torch.optim.SGD(m.parameters(), lr=0.05),
    "SGD+mom":     lambda m: torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9),
    "Adam":        lambda m: torch.optim.Adam(m.parameters(), lr=1e-3),
    "AdamW":       lambda m: torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2),
}
runs = {}
for name, make_opt in configs.items():
    m = fresh_model()
    runs[name] = train_epochs(m, make_opt(m), num_epochs=5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, h in runs.items():
    axes[0].plot(h["train_loss"], marker="o", label=name)
    axes[1].plot(h["val_acc"],    marker="o", label=name)
axes[0].set_title("Train loss");    axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Val accuracy");  axes[1].set_xlabel("epoch"); axes[1].legend()
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

for name, h in runs.items():
    print(f"  {name:8s} final val_acc={h['val_acc'][-1]:.4f}, final train_loss={h['train_loss'][-1]:.4f}")

/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  SGD      final val_acc=0.8735, final train_loss=0.2337
  SGD+mom  final val_acc=0.9130, final train_loss=0.0330
  Adam     final val_acc=0.8865, final train_loss=0.1375
  AdamW    final val_acc=0.8875, final train_loss=0.1380


/tmp/ipykernel_186196/3037196353.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 2. LR-range test

Start at a tiny learning rate and *increase* it exponentially each step.
Plot loss vs. learning rate. The "knee" right before the loss explodes is
your maximum stable LR; pick something 2-3× below it.

In [5]:
def lr_range_test(lr_min=1e-6, lr_max=1.0, num_steps=300):
    torch.manual_seed(0)
    model = make_mlp().to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr_min, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    gamma = (lr_max / lr_min) ** (1 / num_steps)

    lrs, losses = [], []
    it = iter(train_loader)
    for step in range(num_steps):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_loader); x, y = next(it)
        x, y = x.to(DEVICE), y.to(DEVICE)

        opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); opt.step()

        lrs.append(opt.param_groups[0]["lr"])
        losses.append(loss.item())
        for g in opt.param_groups:
            g["lr"] *= gamma
        if loss.item() > 10 or math.isnan(loss.item()):
            break
    return lrs, losses


lrs, losses = lr_range_test()

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(lrs, losses)
ax.set_xlabel("learning rate (log scale)"); ax.set_ylabel("training loss")
ax.set_title("LR-range test on a 1-hidden-layer MLP, SGD+mom 0.9")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()
print(f"swept {len(lrs)} steps; min loss at LR ~ {lrs[int(np.argmin(losses))]:.3g}")

swept 291 steps; min loss at LR ~ 0.0263


/tmp/ipykernel_186196/113428334.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The loss is flat at very low LRs (no progress), drops once the LR becomes
useful, hits a minimum, and finally diverges. Pick a target LR a factor of
2-3 *below* the minimum-loss LR.

## 3. Scheduler ablation

Same optimizer (SGD+momentum), same starting LR. Three schedulers:

- **StepLR** — drop LR by `gamma` every `step_size` epochs.
- **CosineAnnealingLR** — smoothly anneal from `lr_max` to `lr_min`.
- **OneCycleLR** — warm up to `max_lr` then anneal down (per-step).

In [6]:
num_epochs = 8

def run_with_scheduler(make_sched, step="epoch"):
    m = fresh_model()
    opt = torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9)
    sched = make_sched(opt)
    return train_epochs(m, opt, scheduler=sched, num_epochs=num_epochs,
                        step_scheduler=step)


hist_step = run_with_scheduler(
    lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.3)
)
hist_cosine = run_with_scheduler(
    lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)
)
hist_one = run_with_scheduler(
    lambda opt: torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=0.2, steps_per_epoch=len(train_loader), epochs=num_epochs
    ),
    step="batch",
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, h in [("StepLR", hist_step), ("Cosine", hist_cosine), ("OneCycle", hist_one)]:
    axes[0].plot(h["val_acc"], marker="o", label=name)
    axes[1].plot(h["lr"],      marker="o", label=name)
axes[0].set_title("Val accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("LR at end-of-epoch"); axes[1].set_xlabel("epoch"); axes[1].legend()
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

/tmp/ipykernel_186196/570706399.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Note that `OneCycleLR.step()` runs **per batch** (not per epoch). The
`scheduler.get_last_lr()` value reflects this — the LR curve looks
different from the per-epoch StepLR and Cosine schedulers.

## 4. Gradient clipping demo

`clip_grad_norm_` rescales the gradient if its global L2 norm exceeds
`max_norm`. We construct an artificially large gradient and see what
happens.

In [7]:
torch.manual_seed(0)
W = torch.randn(8, 8, requires_grad=True)
loss = (W * 1e3).pow(2).sum()       # artificially huge loss
loss.backward()
print(f"before clip: ‖grad‖ = {W.grad.norm():.2e}")

torch.nn.utils.clip_grad_norm_([W], max_norm=1.0)
print(f"after  clip: ‖grad‖ = {W.grad.norm():.2e}")
print("clip_grad_norm_ rescales the gradient in-place so its norm == max_norm")
print("(if it was below max_norm, nothing happens).")

before clip: ‖grad‖ = 1.67e+07
after  clip: ‖grad‖ = 1.00e+00
clip_grad_norm_ rescales the gradient in-place so its norm == max_norm
(if it was below max_norm, nothing happens).


For LSTMs and Transformers, `max_norm = 1.0` is the standard default.
Without clipping, a single batch with anomalous targets can drive the loss
to NaN.

## Self-test

<details>
<summary>Q1 — Why is the learning rate usually more important than the optimizer choice?</summary>

Within a sane optimizer family, a well-tuned LR almost always beats a
better optimizer with a bad LR. The LR sets the *size* of every update,
which dominates training dynamics.
</details>

<details>
<summary>Q2 — Your <code>OneCycleLR</code> loss looks flat. Likely bug?</summary>

Calling `scheduler.step()` per epoch instead of per batch. `OneCycleLR` is
parameterized by `steps_per_epoch × epochs` and expects a step after every
batch. Fix by moving `scheduler.step()` inside the inner loop.
</details>